# 02. Base (Black-box) Model

GMSC, HC 데이터에 대해 블랙박스 원모형(Base Model)을 학습하고 성능을 평가한다.

- **LightGBM** (기본)
- **XGBoost** (비교)
- Train / Val / Test 분할 (Val은 early stopping 전용)
- 평가: AUC, KS, Log-loss
- Feature Importance 기반 변수 선별
- BB Contributions (귀인 충실도 ground truth)
- 샘플별 Top-k 기여 변수 저장

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle, os
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss

DATA_DIR = '../.data'
RANDOM_STATE = 317

In [2]:
%%time
with open(f'{DATA_DIR}/prepared.pkl', 'rb') as f:
    prepared = pickle.load(f)

datasets = {name: prepared[name] for name in ['GMSC', 'HC']}
for name, d in datasets.items():
    print(f"{name:6s}  X={str(d['X'].shape):>14s}  default={d['y'].mean():.2%}")

GMSC    X=  (150000, 10)  default=6.68%
HC      X=  (307511, 41)  default=8.07%
CPU times: total: 62.5 ms
Wall time: 55.4 ms


---
## 1. Train / Val / Test Split

- Test 20%: 최종 성능 평가
- Val 20% (Train의): Early stopping 전용

In [3]:
splits = {}

for name, d in datasets.items():
    X, y = d['X'], d['y']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)
    
    splits[name] = {
        'X_train': X_train, 'y_train': y_train,
        'X_tr': X_tr, 'y_tr': y_tr,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
    }
    print(f"{name:6s}  Train={len(X_tr):>7,}  Val={len(X_val):>6,}  Test={len(X_test):>6,}")

GMSC    Train= 96,000  Val=24,000  Test=30,000


HC      Train=196,806  Val=49,202  Test=61,503


---
## 2. LightGBM

In [4]:
LGB_PARAMS = dict(
    max_depth=-1,
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,
    verbose=-1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
EARLY_STOPPING_ROUND = 50

In [5]:
%%time
lgb_models = {}

for name, s in splits.items():
    print(f'--- {name} ---')
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(
        s['X_tr'], s['y_tr'],
        eval_set=[(s['X_val'], s['y_val'])],
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUND), lgb.log_evaluation(50)],
    )
    lgb_models[name] = model
    print(f'  best_iteration: {model.best_iteration_}\n')

--- GMSC ---
Training until validation scores don't improve for 50 rounds


[50]	valid_0's binary_logloss: 0.179776


[100]	valid_0's binary_logloss: 0.178618


Early stopping, best iteration is:
[78]	valid_0's binary_logloss: 0.178545
  best_iteration: 78

--- HC ---


Training until validation scores don't improve for 50 rounds


...

[100]	valid_0's binary_logloss: 0.252625


[150]	valid_0's binary_logloss: 0.252376


[200]	valid_0's binary_logloss: 0.252318


[250]	valid_0's binary_logloss: 0.252245


Early stopping, best iteration is:
[247]	valid_0's binary_logloss: 0.25223
  best_iteration: 247

CPU times: total: 47.5 s
Wall time: 6.32 s


In [6]:
def evaluate_model(model, s, label):
    """AUC, KS, Log-loss on test set."""
    prob = model.predict_proba(s['X_test'])[:, 1]
    y = s['y_test']
    auc = roc_auc_score(y, prob)
    ll = log_loss(y, prob)
    from scipy.stats import ks_2samp
    ks = ks_2samp(prob[y == 0], prob[y == 1]).statistic
    return {'Dataset': label, 'AUC': auc, 'KS': ks, 'LogLoss': ll}

In [7]:
lgb_results = [evaluate_model(lgb_models[name], splits[name], name) for name in datasets]
df_lgb = pd.DataFrame(lgb_results).set_index('Dataset').round(4)
print('=== LightGBM (Test Set) ===')
df_lgb

=== LightGBM (Test Set) ===


,AUC,KS,LogLoss
Dataset,,,
GMSC,0.8661,0.5714,0.1772
HC,0.7462,0.3666,0.2497


---
## 3. XGBoost (비교)

In [8]:
XGB_PARAMS = dict(
    max_depth=0,
    n_estimators=1000,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=EARLY_STOPPING_ROUND,
    verbosity=0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

In [9]:
%%time
xgb_models = {}

for name, s in splits.items():
    print(f'--- {name} ---')
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(
        s['X_tr'], s['y_tr'],
        eval_set=[(s['X_val'], s['y_val'])],
        verbose=False,
    )
    xgb_models[name] = model
    print(f'  best_iteration: {model.best_iteration}\n')

--- GMSC ---


  best_iteration: 44

--- HC ---


  best_iteration: 31

CPU times: total: 1min 57s
Wall time: 15.1 s


In [10]:
xgb_results = [evaluate_model(xgb_models[name], splits[name], name) for name in datasets]
df_xgb = pd.DataFrame(xgb_results).set_index('Dataset').round(4)
print('=== XGBoost (Test Set) ===')
df_xgb

=== XGBoost (Test Set) ===


,AUC,KS,LogLoss
Dataset,,,
GMSC,0.8549,0.5564,0.1837
HC,0.7165,0.3170,0.2595


---
## 4. LightGBM vs XGBoost 비교

In [11]:
compare = pd.concat([
    df_lgb.add_prefix('LGB_'),
    df_xgb.add_prefix('XGB_'),
], axis=1)
compare

,LGB_AUC,LGB_KS,LGB_LogLoss,XGB_AUC,XGB_KS,XGB_LogLoss
Dataset,,,,,,
GMSC,0.8661,0.5714,0.1772,0.8549,0.5564,0.1837
HC,0.7462,0.3666,0.2497,0.7165,0.3170,0.2595


---
## 5. Feature Importance & 선별 변수

In [12]:
used_features = {}

for name, model in lgb_models.items():
    imp = pd.DataFrame({
        'feature': model.feature_name_,
        'importance': model.feature_importances_,
    }).sort_values('importance', ascending=False)
    imp['pct'] = (imp['importance'] / imp['importance'].sum()).map('{:.1%}'.format)
    
    selected = imp[imp['importance'] > 0]['feature'].tolist()
    dropped = imp[imp['importance'] == 0]['feature'].tolist()
    used_features[name] = selected
    
    print(f'\n=== {name}: {len(selected)}/{len(imp)} features selected ===')
    print(imp.to_string(index=False))
    if dropped:
        print(f'  Dropped (importance=0): {dropped}')


=== GMSC: 10/10 features selected ===
                             feature  importance   pct
                           DebtRatio         932 19.3%
                       MonthlyIncome         859 17.8%
RevolvingUtilizationOfUnsecuredLines         731 15.1%
                                 age         659 13.6%
     NumberOfOpenCreditLinesAndLoans         549 11.4%
        NumberRealEstateLoansOrLines         291  6.0%
NumberOfTime30-59DaysPastDueNotWorse         243  5.0%
NumberOfTime60-89DaysPastDueNotWorse         215  4.4%
             NumberOfTimes90DaysLate         184  3.8%
                  NumberOfDependents         173  3.6%

=== HC: 41/41 features selected ===
                    feature  importance  pct
               EXT_SOURCE_3        1247 8.1%
                AMT_ANNUITY        1240 8.1%
               EXT_SOURCE_2        1184 7.7%
                 DAYS_BIRTH        1159 7.6%
                 AMT_CREDIT        1122 7.3%
          DAYS_REGISTRATION        1053 6.9%
    

---
## 6. BB Contributions (Ground Truth Attribution)

Test set에 대한 base model의 feature contribution을 산출한다.
- LightGBM: `predict(pred_contrib=True)`
- XGBoost: `predict(pred_contribs=True)`

In [13]:
%%time
bb_contribs = {}

for name in datasets:
    print(f'--- {name} ---')
    s = splits[name]
    X_test = s['X_test']

    # LightGBM: pred_contrib (Regressor 방식 — 마지막 열 = bias)
    lgb_raw = lgb_models[name].predict(X_test, pred_contrib=True)
    lgb_contrib = lgb_raw[:, :-1]
    print(f'  LGB contrib: {lgb_contrib.shape}')

    # XGBoost: pred_contribs
    xgb_dm = xgb.DMatrix(X_test, feature_names=list(X_test.columns))
    xgb_contrib = xgb_models[name].get_booster().predict(
        xgb_dm, pred_contribs=True
    )[:, :-1]
    print(f'  XGB contrib: {xgb_contrib.shape}')

    bb_contribs[name] = {
        'lgb': lgb_contrib,
        'xgb': xgb_contrib,
    }

--- GMSC ---


  LGB contrib: (30000, 10)


  XGB contrib: (30000, 10)
--- HC ---


  LGB contrib: (61503, 41)


  XGB contrib: (61503, 41)
CPU times: total: 4h 3min
Wall time: 21min 31s


---
## 7. 샘플별 Top-k 기여 변수

각 샘플별로 기여도(절대값) 상위 feature를 정리한다.  
LGB contrib 기준, 상위 5개 feature name + score.

In [14]:
%%time
TOP_K = 5
bb_topk = {}

for name in datasets:
    contrib = bb_contribs[name]['lgb']
    feature_names = np.array(lgb_models[name].feature_name_)
    n_samples = contrib.shape[0]
    
    rows = []
    for i in range(n_samples):
        order = np.argsort(np.abs(contrib[i]))[::-1][:TOP_K]
        row = {}
        for rank, idx in enumerate(order, 1):
            row[f'top{rank}_feature'] = feature_names[idx]
            row[f'top{rank}_score'] = round(float(contrib[i, idx]), 6)
        rows.append(row)
    
    bb_topk[name] = pd.DataFrame(rows)
    print(f'\n=== {name}: Top-{TOP_K} attribution per sample ===')
    print(bb_topk[name].head(10).to_string(index=False))


=== GMSC: Top-5 attribution per sample ===
                        top1_feature  top1_score                         top2_feature  top2_score                    top3_feature  top3_score                         top4_feature  top4_score                         top5_feature  top5_score
NumberOfTime30-59DaysPastDueNotWorse    1.824925 RevolvingUtilizationOfUnsecuredLines    0.987218                             age    0.153719      NumberOfOpenCreditLinesAndLoans   -0.118530              NumberOfTimes90DaysLate   -0.107199
RevolvingUtilizationOfUnsecuredLines   -0.606409                                  age   -0.297308 NumberOfOpenCreditLinesAndLoans   -0.240229 NumberOfTime30-59DaysPastDueNotWorse   -0.223624              NumberOfTimes90DaysLate   -0.111371
     NumberOfOpenCreditLinesAndLoans   -0.179483 NumberOfTime30-59DaysPastDueNotWorse   -0.178428                             age    0.173150 RevolvingUtilizationOfUnsecuredLines    0.166305                        MonthlyIncome    0.130


=== HC: Top-5 attribution per sample ===
          top1_feature  top1_score           top2_feature  top2_score              top3_feature  top3_score           top4_feature  top4_score                top5_feature  top5_score
          EXT_SOURCE_3   -0.627720           EXT_SOURCE_2   -0.403872                AMT_CREDIT   -0.202071        DAYS_ID_PUBLISH   -0.196513             AMT_GOODS_PRICE    0.143338
          EXT_SOURCE_2   -0.342916      DAYS_REGISTRATION    0.206263          AMT_INCOME_TOTAL   -0.185026        FLAG_DOCUMENT_3   -0.158234 REGION_RATING_CLIENT_W_CITY   -0.129545
          EXT_SOURCE_3   -0.448634             DAYS_BIRTH   -0.186286           AMT_GOODS_PRICE    0.096814        FLAG_DOCUMENT_3   -0.093567             DAYS_ID_PUBLISH   -0.089499
          EXT_SOURCE_3   -0.619121             AMT_CREDIT   -0.534333              EXT_SOURCE_2    0.389964        AMT_GOODS_PRICE    0.293439      DAYS_LAST_PHONE_CHANGE    0.115579
          EXT_SOURCE_3   -0.553760 DAYS_LAS

---
## Save

Base models, splits, BB contribs, 선별 변수, Top-k 기여 변수를 저장한다.

In [15]:
%%time
base_models = {
    name: {
        'lgb': lgb_models[name],
        'xgb': xgb_models[name],
        'splits': splits[name],
        'bb_contribs': bb_contribs[name],
        'used_features': used_features[name],
        'bb_topk': bb_topk[name],
    }
    for name in datasets
}

out_path = f'{DATA_DIR}/base_models.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(base_models, f)

size_mb = os.path.getsize(out_path) / 1024 / 1024
print(f'Saved to {out_path} ({size_mb:.1f} MB)')
for name in datasets:
    bm = base_models[name]
    print(f"  {name}: used_features={len(bm['used_features'])}, "
          f"bb_topk={bm['bb_topk'].shape}")

Saved to ../.data/base_models.pkl (302.8 MB)
  GMSC: used_features=10, bb_topk=(30000, 10)
  HC: used_features=41, bb_topk=(61503, 10)
CPU times: total: 1.58 s
Wall time: 289 ms
